In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
FIELDVARS  = CONFIGS['experiments']['sr']['runs']['sr_atm']['fieldvars']
SEEDS      = CONFIGS['experiments']['nn']['seeds']
SPLIT      = 'test'
MINCUBESAMPLES = 100

SRFUNCTIONS = {
    'cube':lambda x:x**3,'square':lambda x:x**2,'neg':lambda x:-x,
    'sqrt':np.sqrt,'exp':np.exp,'log':np.log,'abs':np.abs,
    'sin':np.sin,'cos':np.cos,'max':np.maximum,'min':np.minimum,
    '_safepow':lambda a,b:np.abs(a)**b}

import re
def _prepare_form(form):
    return re.sub(r'(\w+)\^(\w+)',r'_safepow(\1,\2)',form)

def eval_form(form,columns,constants):
    ns = dict(SRFUNCTIONS,__builtins__={})
    ns.update(columns)
    ns.update(constants)
    out = eval(_prepare_form(form),ns)
    if np.ndim(out)==0:
        n = len(next(v for v in columns.values() if hasattr(v,'__len__')))
        out = np.full(n,float(out))
    return np.asarray(out,dtype=float)

In [ ]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
MEAN = STATS['tp_mean']
STD  = STATS['tp_std']
ZMIN = (0.0 - MEAN) / STD

with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime = ds.sizes['time']
    nsig  = ds.sizes.get('sig',1)
    dsig  = ds['dsig'].values
    fields = np.stack([ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS],axis=1)
    surfmask = ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig) if 'surfmask' in ds else None
    flat = lambda v: ds[v].transpose('time','lat','lon').values.ravel() if 'time' in ds[v].dims else np.tile(ds[v].values,(ntime,1,1)).ravel()
    lfraw = flat('lf')
    shfraw = flat('shf')
    lhfraw = flat('lhf')
    blraw = flat('bl') if 'bl' in ds else np.zeros(ntime*ds.sizes['lat']*ds.sizes['lon'])

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obsraw = ds['tp'].transpose('time','lat','lon').values.ravel()

kernels = []
for seed in SEEDS:
    with xr.open_dataset(os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc'),engine='h5netcdf') as ds:
        kernels.append(ds.k.values)
meankernel = np.mean(kernels,axis=0)
weighted = fields * meankernel[None,:,:] * dsig[None,None,:]
if surfmask is not None:
    weighted = weighted * surfmask[:,None,:]
integrals = weighted.sum(axis=2)
rhraw,thetaeraw,thetaestarraw = integrals[:,0],integrals[:,1],integrals[:,2]

valid = np.isfinite(rhraw) & np.isfinite(thetaeraw) & np.isfinite(thetaestarraw) & np.isfinite(obsraw)
rh,thetae,thetaestar = rhraw[valid],thetaeraw[valid],thetaestarraw[valid]
lf,shf,lhf = lfraw[valid],shfraw[valid],lhfraw[valid]
bl = blraw[valid]
obs = obsraw[valid]
landmask  = lf > 0.5
oceanmask = lf < 0.5
print(f'Loaded {valid.sum():,} valid samples ({landmask.sum():,} land, {oceanmask.sum():,} ocean)')

In [ ]:
regdf = pd.read_csv(os.path.join(MODELSDIR,'sr','optimized_equations.csv'))
REGISTRY = {row['name']:dict(form=row['form'],constants=json.loads(row['constants']),
                              train_loss=row['train_loss'],valid_loss=row['valid_loss'])
             for _,row in regdf.iterrows()}
SRMODELS = CONFIGS['experiments']['sr']['optimizedeqs']
ORDER  = [name for name in SRMODELS if name in REGISTRY]
LABELS = {name:SRMODELS[name]['description'] for name in ORDER}
COLORS = {name:SRMODELS[name]['color'] for name in ORDER}

def get_columns(**overrides):
    cols = {'rh':rh,'thetae':thetae,'thetaestar':thetaestar,
            'lf':lf,'shf':shf,'lhf':lhf,'bl':bl}
    cols.update(overrides)
    for eqname,entry in REGISTRY.items():
        if eqname in overrides:
            continue
        cols[eqname] = eval_form(entry['form'],cols,entry['constants'])
    return cols

def predict_eq(name,columns):
    entry = REGISTRY[name]
    raw = eval_form(entry['form'],columns,entry['constants'])
    z = ZMIN + np.maximum(raw,0.0)
    return np.maximum(np.expm1(z * STD + MEAN),0.0)

print(f'Loaded {len(ORDER)} optimized equations: {[LABELS[n] for n in ORDER]}')

## Physical Constraints and Analytical Derivatives

We define four monotonicity constraints on partial derivatives of precipitation with respect to predictors:

| Constraint | Requirement | Physical Meaning |
|:-----------|:------------|:-----------------|
| PC1 | $\partial P / \partial B_L \geq 0$ | More buoyancy $\rightarrow$ more rain |
| PC2 | $\partial P / \partial \widehat{\mathrm{RH}} \geq 0$ | More moisture $\rightarrow$ more rain |
| PC3 | $\partial P / \partial \widehat{\theta_e} \geq 0$ | More energy $\rightarrow$ more rain |
| PC4 | $\partial P / \partial \widehat{\theta_e^*} \leq 0$ | More stability $\rightarrow$ less rain |

Since $dP/dz \geq 0$ in the active precipitation region, checking $\text{sign}(\partial P / \partial x_i)$ reduces to checking $\text{sign}(\partial z / \partial x_i)$.

For ERA5 observations (no closed-form $z$), we use hypercube binning. For SR models, we compute $\partial z / \partial x_i$ analytically at every sample.

In [ ]:
CONSTRAINTS = {
    'PC1':{
        'label':r'$\partial P/\partial \mathrm{B_L} \geq 0$',
        'target_var':'bl',
        'expected_sign':1,
        'equations':['sr_bl_eq']},
    'PC2':{
        'label':r'$\partial P/\partial \widehat{\mathrm{RH}} \geq 0$',
        'target_var':'rh',
        'expected_sign':1,
        'equations':['sr_atm_eq','sr_sfc_eq','sr_all_eq','sr_all_pc_eq']},
    'PC3':{
        'label':r'$\partial P/\partial \widehat{\theta_e} \geq 0$',
        'target_var':'thetae',
        'expected_sign':1,
        'equations':['sr_atm_eq','sr_sfc_eq','sr_all_eq','sr_all_pc_eq']},
    'PC4':{
        'label':r'$\partial P/\partial \widehat{\theta_e^*} \leq 0$',
        'target_var':'thetaestar',
        'expected_sign':-1,
        'equations':['sr_atm_eq','sr_sfc_eq','sr_all_eq','sr_all_pc_eq']}}

ERA5_CONSTRAINTS = {
    'PC1':{'target_var':'bl','expected_sign':1,'other_vars':[]},
    'PC2':{'target_var':'rh','expected_sign':1,'other_vars':['thetae','thetaestar']},
    'PC3':{'target_var':'thetae','expected_sign':1,'other_vars':['rh','thetaestar']},
    'PC4':{'target_var':'thetaestar','expected_sign':-1,'other_vars':['rh','thetae']}}

PCLABELS = {k:v['label'] for k,v in CONSTRAINTS.items()}

In [ ]:
def calc_dz(eqname,constants=None):
    if eqname == 'sr_bl_eq':
        c = constants or REGISTRY['sr_bl_eq']['constants']
        return {'bl':3*(bl+c['a'])**2}
    atm = REGISTRY['sr_atm_eq']['constants']
    a_atm,b_atm,c_atm = atm['a'],atm['b'],atm['c']
    arg = thetae - b_atm*thetaestar - c_atm
    rh_branch = rh >= arg
    buoy_branch = ~rh_branch
    dz_drh = np.where(rh_branch,3*a_atm*rh**2,0.0)
    dz_dthetae_atm = np.where(buoy_branch,3*a_atm*arg**2,0.0)
    dz_dthetaestar = np.where(buoy_branch,-b_atm*3*a_atm*arg**2,0.0)
    if eqname == 'sr_atm_eq':
        return {'rh':dz_drh,'thetae':dz_dthetae_atm,'thetaestar':dz_dthetaestar}
    if eqname == 'sr_sfc_eq':
        return {'rh':dz_drh,'thetae':dz_dthetae_atm,'thetaestar':dz_dthetaestar}
    c = constants or REGISTRY[eqname]['constants']
    if eqname == 'sr_all_eq':
        return {'rh':dz_drh,'thetae':dz_dthetae_atm+(c['b']-lf)**3,'thetaestar':dz_dthetaestar}
    if eqname == 'sr_all_pc_eq':
        return {'rh':dz_drh,'thetae':dz_dthetae_atm+(c['b']-lf)**3-(c['b']-1)**3,'thetaestar':dz_dthetaestar}
    raise ValueError(f'Unknown equation: {eqname}')

In [ ]:
print('=== Analytical Partial Derivatives dz/dx_i ===')
print()
print('--- SR-BL: z = cube(bl + a) + b ---')
print('  dz/d(bl) = 3*(bl + a)^2           >= 0 always     [PC1 OK]')
print()
print('--- SR-ATM: z = a_atm * cube(max(rh, thetae - b_atm*thetaestar - c_atm)) ---')
print('  Let M = max(rh, thetae - b_atm*thetaestar - c_atm)')
print('  RH branch (rh >= arg):')
print('    dz/d(rh)         = 3*a_atm*rh^2           >= 0  [PC2 OK]')
print('    dz/d(thetae)     = 0                       >= 0  [PC3 OK]')
print('    dz/d(thetaestar) = 0                       <= 0  [PC4 OK]')
print('  Buoyancy branch (rh < arg):')
print('    dz/d(rh)         = 0                       >= 0  [PC2 OK]')
print('    dz/d(thetae)     = 3*a_atm*arg^2           >= 0  [PC3 OK]')
print('    dz/d(thetaestar) = -b_atm*3*a_atm*arg^2   <= 0  [PC4 OK]')
print()
print('--- SR-SFC: z = SR-ATM + a*shf*(b - lf) + c*lhf ---')
print('  dz/d(rh), dz/d(thetae), dz/d(thetaestar) identical to SR-ATM')
print('  (surface terms do not depend on kernel-integrated variables)')
print()
print('--- SR-ALL: z = SR-ATM + (thetae + a*shf)*cube(b - lf) + c ---')
print('  RH branch:')
print('    dz/d(rh)         = 3*a_atm*rh^2                    >= 0  [PC2 OK]')
print('    dz/d(thetae)     = cube(b - lf)                    SIGN UNCERTAIN  [PC3 ???]')
print('    dz/d(thetaestar) = 0                                <= 0  [PC4 OK]')
print('  Buoyancy branch:')
print('    dz/d(rh)         = 0                                >= 0  [PC2 OK]')
print('    dz/d(thetae)     = 3*a_atm*arg^2 + cube(b - lf)   FIRST >= 0, SECOND UNCERTAIN  [PC3 ???]')
print('    dz/d(thetaestar) = -b_atm*3*a_atm*arg^2            <= 0  [PC4 OK]')
print()
print('  PC3 violation: cube(b - lf) < 0 when lf > b.')
print(f'  With b = {REGISTRY["sr_all_eq"]["constants"]["b"]:.2f}, all land points (lf ~ 1) violate in the RH branch.')
print()
print('--- SR-ALL-PC: z = SR-ATM + (thetae + a*shf)*cube(b - lf) - cube(b - 1)*thetae + c ---')
print('  dz/d(thetae) = [same as SR-ALL] + cube(b - lf) - cube(b - 1)')
print('  Since min(cube(b - lf)) = cube(b - 1) at lf = 1:')
print('    cube(b - lf) - cube(b - 1) >= 0 for all lf in [0,1]')
print('  Both branches: dz/d(thetae) >= sum of non-negative terms  [PC3 OK]')
print('  All other derivatives unchanged from SR-ALL  [PC2, PC4 OK]')

## ERA5 Physical Constraint Compliance (Hypercube Test)

In [ ]:
NCUBES = [3,4,5,6,7]

def hypercube_test(target_var,expected_sign,other_vars,ncubes,mask=None):
    featurevals = {'rh':rh,'thetae':thetae,'thetaestar':thetaestar,
                   'lf':lf,'shf':shf,'lhf':lhf,'bl':bl}
    targetvals = featurevals[target_var]
    othervalslist = [featurevals[v] for v in other_vars]
    nother = len(other_vars)
    if mask is None:
        mask = np.ones(len(targetvals),dtype=bool)
    edges = [np.linspace(np.percentile(v[mask],1),np.percentile(v[mask],99),ncubes+1) for v in othervalslist]
    bins  = [np.clip(np.digitize(v,e)-1,0,ncubes-1) for v,e in zip(othervalslist,edges)]
    nsatisfied,ntested = 0,0
    cubeidx = bins[0].copy()
    for i in range(1,nother):
        cubeidx = cubeidx * ncubes + bins[i]
    for cidx in range(ncubes**nother):
        sel = mask & (cubeidx == cidx)
        if sel.sum() < MINCUBESAMPLES:
            continue
        x,y = targetvals[sel],obs[sel]
        if len(x) < 2:
            continue
        xc = x - x.mean()
        slope = np.dot(xc,y) / (np.dot(xc,xc) + 1e-12)
        ntested += 1
        if (expected_sign >= 0 and slope >= 0) or (expected_sign < 0 and slope <= 0):
            nsatisfied += 1
    return nsatisfied,ntested

era5rows = []
for pcname,pc in ERA5_CONSTRAINTS.items():
    for region,mask in [('Land',landmask),('Ocean',oceanmask),('All',None)]:
        if not pc['other_vars']:
            m = mask if mask is not None else np.ones(len(bl),dtype=bool)
            xc = bl[m] - bl[m].mean()
            slope = np.dot(xc,obs[m]) / (np.dot(xc,xc) + 1e-12)
            for n in NCUBES:
                era5rows.append({'PC':pcname,'Region':region,'N':n,
                                 'Satisfied':1 if slope >= 0 else 0,'Tested':1,
                                 'Pct':100.0 if slope >= 0 else 0.0})
        else:
            for n in NCUBES:
                sat,tot = hypercube_test(pc['target_var'],pc['expected_sign'],pc['other_vars'],n,mask)
                era5rows.append({'PC':pcname,'Region':region,'N':n,
                                 'Satisfied':sat,'Tested':tot,
                                 'Pct':sat/max(tot,1)*100})

era5df = pd.DataFrame(era5rows)
era5pivot = era5df.pivot_table(index=['PC','Region'],columns='N',values='Pct')
era5pivot['Average'] = era5pivot.mean(axis=1)
era5pivot = era5pivot.reindex(
    pd.MultiIndex.from_product([['PC1','PC2','PC3','PC4'],['Land','Ocean','All']],
                               names=['PC','Region']))
era5pivot.style.format('{:.1f}%').set_caption(
    f'ERA5 physical constraint satisfaction (%, {SPLIT} split, hypercube binning, min {MINCUBESAMPLES} samples/cube)')

## Ocean PC2 Diagnostic

ERA5 shows some PC2 violations over ocean in the hypercube test. This figure examines whether those violations are a data artifact (occurring in dry, stable regimes with little precipitation where the monotonicity signal is weak) or reflect real physics.

In [ ]:
NCUBE = 5
pc = ERA5_CONSTRAINTS['PC2']
featurevals = {'rh':rh,'thetae':thetae,'thetaestar':thetaestar,
               'lf':lf,'shf':shf,'lhf':lhf,'bl':bl}
targetvals = featurevals[pc['target_var']]
othervalslist = [featurevals[v] for v in pc['other_vars']]
nother = len(pc['other_vars'])

cuberows = []
for region,mask in [('Land',landmask),('Ocean',oceanmask)]:
    edges = [np.linspace(np.percentile(v[mask],1),np.percentile(v[mask],99),NCUBE+1) for v in othervalslist]
    bins = [np.clip(np.digitize(v,e)-1,0,NCUBE-1) for v,e in zip(othervalslist,edges)]
    cubeidx = bins[0].copy()
    for i in range(1,nother):
        cubeidx = cubeidx * NCUBE + bins[i]
    for cidx in range(NCUBE**nother):
        sel = mask & (cubeidx == cidx)
        if sel.sum() < MINCUBESAMPLES:
            continue
        x,y = targetvals[sel],obs[sel]
        xc = x - x.mean()
        slope = np.dot(xc,y) / (np.dot(xc,xc) + 1e-12)
        cuberows.append({
            'region':region,
            'slope':slope,
            'satisfied':slope >= 0,
            'nsamples':int(sel.sum()),
            'rh_mean':rh[sel].mean(),
            'precip_mean':obs[sel].mean(),
            'stability':thetaestar[sel].mean() - thetae[sel].mean()})

cubedf = pd.DataFrame(cuberows)
ocean = cubedf[cubedf['region']=='Ocean']
land = cubedf[cubedf['region']=='Land']
print(f'Ocean cubes: {len(ocean)} ({ocean.satisfied.sum()} satisfy PC2, '
      f'{(~ocean.satisfied).sum()} violate)')
print(f'Land cubes:  {len(land)} ({land.satisfied.sum()} satisfy PC2, '
      f'{(~land.satisfied).sum()} violate)')

In [ ]:
fig,axs = pplt.subplots(ncols=3,figwidth=8,refheight=2,sharey=False)

binshist = np.linspace(min(ocean['slope'].min(),-0.5),ocean['slope'].max(),30)
axs[0].hist(ocean[ocean['satisfied']]['slope'].values,bins=binshist,
            color='#2355a1',alpha=0.7,label=f'Satisfied (n={ocean.satisfied.sum()})')
axs[0].hist(ocean[~ocean['satisfied']]['slope'].values,bins=binshist,
            color='#C44E52',alpha=0.7,label=f'Violated (n={(~ocean.satisfied).sum()})')
axs[0].axvline(0,color='k',ls='--',lw=0.8)
axs[0].format(xlabel=r'$\partial P / \partial \widehat{\mathrm{RH}}$ slope',
              ylabel='Number of cubes',title='Slope magnitude')
axs[0].legend(loc='ur',fontsize=7,ncols=1)

axs[1].scatter(ocean[ocean['satisfied']]['rh_mean'],ocean[ocean['satisfied']]['stability'],
               c='#2355a1',s=20,alpha=0.6,label='Satisfied',zorder=2)
axs[1].scatter(ocean[~ocean['satisfied']]['rh_mean'],ocean[~ocean['satisfied']]['stability'],
               c='#C44E52',s=40,alpha=0.8,label='Violated',marker='x',zorder=3)
axs[1].format(xlabel=r'Cube mean $\widehat{\mathrm{RH}}$',
              ylabel=r'Cube mean $\widehat{\theta_e^*} - \widehat{\theta_e}$',
              title='Thermodynamic regime')
axs[1].legend(loc='ur',fontsize=7,ncols=1)

axs[2].scatter(ocean[ocean['satisfied']]['rh_mean'],ocean[ocean['satisfied']]['precip_mean'],
               c='#2355a1',s=20,alpha=0.6,label='Satisfied',zorder=2)
axs[2].scatter(ocean[~ocean['satisfied']]['rh_mean'],ocean[~ocean['satisfied']]['precip_mean'],
               c='#C44E52',s=40,alpha=0.8,label='Violated',marker='x',zorder=3)
axs[2].format(xlabel=r'Cube mean $\widehat{\mathrm{RH}}$',ylabel='Cube mean precip (mm)',
              title='Precipitation regime')
axs[2].legend(loc='ur',fontsize=7,ncols=1)

axs.format(abc=True,titleloc='l')
fig.save('../figs/fig_S2.jpg')

## SR Model Physical Constraint Compliance (Analytical Derivatives)

In [ ]:
modelresults = {}
for name in ORDER:
    modelresults[name] = {}
    derivs = calc_dz(name)
    for pcname,pc in CONSTRAINTS.items():
        if name not in pc['equations']:
            modelresults[name][pcname] = {'Land':np.nan,'Ocean':np.nan,'All':np.nan}
            continue
        dz = derivs[pc['target_var']]
        if pc['expected_sign'] >= 0:
            modelresults[name][pcname] = {
                'Land':np.mean(dz[landmask] >= 0)*100,
                'Ocean':np.mean(dz[oceanmask] >= 0)*100,
                'All':np.mean(dz >= 0)*100}
        else:
            modelresults[name][pcname] = {
                'Land':np.mean(dz[landmask] <= 0)*100,
                'Ocean':np.mean(dz[oceanmask] <= 0)*100,
                'All':np.mean(dz <= 0)*100}

srrows = []
for name in ORDER:
    for region in ['Land','Ocean','All']:
        row = {'Model':LABELS[name],'Region':region}
        for pcname in CONSTRAINTS:
            row[PCLABELS[pcname]] = modelresults[name][pcname][region]
        srrows.append(row)

srdf = pd.DataFrame(srrows).set_index(['Model','Region'])

def fmt(v):
    if np.isnan(v):
        return ''
    return f'{v:.1f}%'

srdf.style.format(fmt).set_caption(
    f'SR model physical constraint satisfaction (%, {SPLIT} split, analytical derivatives at all samples)')

In [ ]:
srall = REGISTRY['sr_all_eq']['constants']
atm = REGISTRY['sr_atm_eq']['constants']
print('PC3 Violation Mechanism in SR-ALL')
print('=================================')
print(f'SR-ALL form: {REGISTRY["sr_all_eq"]["form"]}')
print(f'SR-ALL constants: {srall}')
print(f'SR-ATM constants: {atm}')
print()
print(f'dz/d(thetae) includes the term cube(b - lf) where b = {srall["b"]:.2f}')
print(f'  Over ocean (lf ~ 0): cube({srall["b"]:.2f} - 0) = {srall["b"]**3:.4f} > 0  [OK]')
print(f'  Over land  (lf ~ 1): cube({srall["b"]:.2f} - 1) = {(srall["b"]-1)**3:.4f} < 0  [VIOLATION]')
print()
print('In the RH branch, cube(b-lf) is the ONLY contribution to dz/d(thetae),')
print('so all land RH-branch samples violate PC3.')
print(f'In the buoyancy branch, the positive 3*a_atm*arg^2 term can compensate,')
print('but not always — hence 62.1% satisfaction over land (vs 100% over ocean).')

## Enforcing PC3: Constructing SR-ALL-PC

The structural fix subtracts $\text{cube}(b-1) \cdot \hat{\theta}_e$ from SR-ALL, shifting the $\hat{\theta}_e$ coefficient from $\text{cube}(b - \text{LF})$ to $\text{cube}(b - \text{LF}) - \text{cube}(b - 1) \geq 0$ for all $\text{LF} \in [0,1]$.

This is smoother than $\max$ clipping (no discontinuous derivative at $\text{LF} = b$), preserves the cubic structure, and at $\text{LF} = 1$ the $\hat{\theta}_e$ correction vanishes exactly.

In [ ]:
PCFORM = SRMODELS['sr_all_pc_eq']['form']
PCNAME = 'sr_all_pc_eq'
PCLABEL = SRMODELS['sr_all_pc_eq']['description']

print(f'SR-ALL form:    {REGISTRY["sr_all_eq"]["form"]}')
print(f'SR-ALL-PC form: {PCFORM}')
print()
print('Modification: thetae*cube(b-lf) -> thetae*(cube(b-lf) - cube(b-1))')
print(f'  At lf=0: cube(b) - cube(b-1) = {srall["b"]**3 - (srall["b"]-1)**3:.4f} > 0')
print(f'  At lf=1: cube(b-1) - cube(b-1) = 0')
print(f'  Monotonically decreasing from lf=0 to lf=1, always non-negative.')

In [ ]:
from scipy.optimize import minimize

y = (np.log1p(obs) - MEAN) / STD
cols = get_columns()

constantnames = ['a','b','c']
initdict = SRMODELS['sr_all_pc_eq'].get('init',REGISTRY['sr_all_eq']['constants'])
initparams = np.array([initdict[c] for c in constantnames])

def objective(params):
    constants = dict(zip(constantnames,params))
    raw = eval_form(PCFORM,cols,constants)
    pred = ZMIN + np.maximum(raw,0.0)
    return float(np.mean((pred - y)**2))

nrestarts = 50
rng = np.random.default_rng(42)
allresults = []
allinits = [initparams] + [rng.uniform(-5,5,len(constantnames)) for _ in range(nrestarts-1)]
for i,x0 in enumerate(allinits):
    res = minimize(objective,x0,method='L-BFGS-B',options={'maxiter':10000,'ftol':1e-14,'gtol':1e-10})
    allresults.append(res)
bestres = min(allresults,key=lambda r:r.fun)
pcconstants = dict(zip(constantnames,bestres.x))
pcconstants_rounded = {k:round(float(v),2) for k,v in pcconstants.items()}

print(f'Optimized on {SPLIT} split ({valid.sum():,} samples, {nrestarts} restarts)')
print(f'  Constants (raw):     {{", ".join(f"{k}={v:.6f}" for k,v in pcconstants.items())}}')
print(f'  Constants (rounded): {{", ".join(f"{k}={v:.2f}" for k,v in pcconstants_rounded.items())}}')
print(f'  MSE (z-space): {bestres.fun:.6f}')
print()
print(f'NOTE: These constants are optimized on the {SPLIT} split only.')
print('For final constants, run on train+valid via the pipeline:')
print('  python -m scripts.models.sr.optimize --equations sr_all_pc_eq')

In [ ]:
derivs_pc = calc_dz('sr_all_pc_eq',pcconstants_rounded)
pcresults = {}
for pcname,pc in CONSTRAINTS.items():
    if PCNAME not in pc['equations']:
        pcresults[pcname] = {'Land':np.nan,'Ocean':np.nan,'All':np.nan}
        continue
    dz = derivs_pc[pc['target_var']]
    if pc['expected_sign'] >= 0:
        pcresults[pcname] = {
            'Land':np.mean(dz[landmask] >= 0)*100,
            'Ocean':np.mean(dz[oceanmask] >= 0)*100,
            'All':np.mean(dz >= 0)*100}
    else:
        pcresults[pcname] = {
            'Land':np.mean(dz[landmask] <= 0)*100,
            'Ocean':np.mean(dz[oceanmask] <= 0)*100,
            'All':np.mean(dz <= 0)*100}

print(f'SR-ALL-PC constraint satisfaction (analytical derivatives, {SPLIT} split):')
print(f'{"":>6} {"":>38} {"SR-ALL":>10} {"SR-ALL-PC":>10} {"Delta":>8}')
for pcname in CONSTRAINTS:
    label = CONSTRAINTS[pcname]['label']
    for region in ['Land','Ocean','All']:
        old = modelresults.get('sr_all_eq',{}).get(pcname,{}).get(region,np.nan)
        new = pcresults[pcname][region]
        delta = new - old if not np.isnan(old) else np.nan
        tag = f'{pcname} {region}'
        oldstr = f'{old:.1f}%' if not np.isnan(old) else 'N/A'
        newstr = f'{new:.1f}%' if not np.isnan(new) else 'N/A'
        deltastr = f'{delta:+.1f}%' if not np.isnan(delta) else ''
        print(f'{tag:>18} {label:>22}  {oldstr:>10}  {newstr:>10}  {deltastr:>8}')

In [ ]:
pred_srall = predict_eq('sr_all_eq',cols)

def predict_pc(columns):
    raw = eval_form(PCFORM,columns,pcconstants_rounded)
    z = ZMIN + np.maximum(raw,0.0)
    return np.maximum(np.expm1(z * STD + MEAN),0.0)

pred_pc = predict_pc(cols)

r2all   = 1 - np.mean((pred_srall - obs)**2) / np.var(obs)
r2land  = 1 - np.mean((pred_srall[landmask] - obs[landmask])**2) / np.var(obs[landmask])
r2ocean = 1 - np.mean((pred_srall[oceanmask] - obs[oceanmask])**2) / np.var(obs[oceanmask])

r2pc_all   = 1 - np.mean((pred_pc - obs)**2) / np.var(obs)
r2pc_land  = 1 - np.mean((pred_pc[landmask] - obs[landmask])**2) / np.var(obs[landmask])
r2pc_ocean = 1 - np.mean((pred_pc[oceanmask] - obs[oceanmask])**2) / np.var(obs[oceanmask])

print(f'Accuracy comparison ({SPLIT} split):')
print(f'{"Model":<14} {"R\u00b2 all":>8} {"R\u00b2 land":>9} {"R\u00b2 ocean":>10}')
print(f'{"SR-ALL":<14} {r2all:>8.4f} {r2land:>9.4f} {r2ocean:>10.4f}')
print(f'{"SR-ALL-PC":<14} {r2pc_all:>8.4f} {r2pc_land:>9.4f} {r2pc_ocean:>10.4f}')
print(f'{"Delta":<14} {r2pc_all-r2all:>+8.4f} {r2pc_land-r2land:>+9.4f} {r2pc_ocean-r2ocean:>+10.4f}')

## Summary Table

In [ ]:
era5avg = {pcname:era5df[era5df['PC']==pcname].groupby('Region')['Pct'].mean().to_dict()
           for pcname in ERA5_CONSTRAINTS}

summrows = []
for region in ['Land','Ocean','All']:
    row = {'Source':'ERA5','Region':region}
    for pcname in CONSTRAINTS:
        row[PCLABELS[pcname]] = era5avg.get(pcname,{}).get(region,np.nan)
    summrows.append(row)

for name in ORDER:
    for region in ['Land','Ocean','All']:
        row = {'Source':LABELS[name],'Region':region}
        for pcname in CONSTRAINTS:
            row[PCLABELS[pcname]] = modelresults[name][pcname][region]
        summrows.append(row)

for region in ['Land','Ocean','All']:
    row = {'Source':PCLABEL,'Region':region}
    for pcname in CONSTRAINTS:
        row[PCLABELS[pcname]] = pcresults[pcname][region]
    summrows.append(row)

summdf = pd.DataFrame(summrows).set_index(['Source','Region'])
summdf.style.format(fmt).set_caption(
    f'Physical constraint satisfaction (%, {SPLIT} split). '
    f'ERA5 via hypercube binning; SR models via analytical derivatives.')

In [ ]:
print('Updated entry for configs.json (experiments.sr.optimizedeqs):')
print()
print(json.dumps({PCNAME:{
    'runfrom':'sr_all',
    'refcomplexity':None,
    'form':PCFORM,
    'init':{k:round(float(v),2) for k,v in pcconstants.items()},
    'color':'#8B0000',
    'description':PCLABEL}},indent=4))
print()
print('To optimize final constants on train+valid:')
print('  python -m scripts.models.sr.optimize --equations sr_all_pc_eq')
print()
print('To generate predictions:')
print('  python -m scripts.models.sr.optimize --equations sr_all_pc_eq --predict-only --splits test')